# 07 — Annotation vidéo avec GPT-6 Luna

Ce notebook orchestre l'annotation temporelle sans recopier durablement les images de la vidéo. Pour chaque cas, Luna reçoit `previous`, `current` et `next`, puis renvoie soit une boîte serrée dans `current`, soit une décision négative.

Le workflow est volontairement séparé en cinq actions :

1. **plan** : vérification locale, gratuite ;
2. **submit** : upload temporaire des images et lancement du Batch API ;
3. **status** : lecture de l'avancement ;
4. **collect** : téléchargement et conversion en labels YOLO ;
5. **cleanup** : suppression des fichiers distants après vérification locale.

> `Run all` ne lance aucun appel payant : `SUBMIT_BATCH` et `DELETE_REMOTE_FILES` restent à `False` par défaut.

In [ ]:
import json
import os
from pathlib import Path

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Commencer par le pilote. Ne passer au fichier full qu'après inspection.
CONFIG_PATH = PROJECT_ROOT / 'configs/luna_video_pilot.toml'
# CONFIG_PATH = PROJECT_ROOT / 'configs/luna_video_full.toml'

from pitchlens.data.luna_batch import (
    build_plan,
    cleanup_remote_files,
    collect_batch,
    load_config,
    refresh_status,
    submit_batch,
)

config = load_config(CONFIG_PATH, PROJECT_ROOT)
config

## 1. Construire et lire le plan

Cette étape n'utilise ni la clé ni l'API. Le plan est immuable : changer la vidéo, le modèle ou l'échantillonnage exige un autre dossier `output`.

In [ ]:
plan = build_plan(config)
display(Markdown(
    f"**Vidéo :** `{plan['source_video']}`  \
"
    f"**Cas à annoter :** {plan['case_count']:,}  \
"
    f"**Images distantes uniques :** {plan['unique_uploaded_frame_count']:,}  \
"
    f"**Modèle :** `{plan['model']}` — raisonnement `{plan['reasoning_effort']}` — détail `{plan['image_detail']}`  \
"
    f"**Sortie locale :** `{config.output_root}`"
))

## 2. Vérifier la clé locale

Le fichier `.env` doit contenir `OPENAI_API_KEY=...`. La valeur n'est jamais affichée.

In [ ]:
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env', override=False)
key_configured = bool(os.getenv('OPENAI_API_KEY'))
display(Markdown(
    '✅ Clé API détectée.' if key_configured
    else '❌ Clé absente : complète le fichier `.env` avant la soumission.'
))

## 3. Soumettre le batch

Cette cellule est la seule qui envoie toutes les images et lance le calcul payant. Elle est reprenable : une coupure conserve chaque `file_id` déjà uploadé.

In [ ]:
SUBMIT_BATCH = False

if SUBMIT_BATCH:
    state = submit_batch(config)
    display(Markdown(f"Batch `{state['batch_id']}` : **{state['batch_status']}**"))
else:
    display(Markdown("Soumission désactivée. Passe `SUBMIT_BATCH = True` uniquement quand tu es prêt."))

## 4. Consulter l'avancement, puis collecter

`status` ne relance aucun calcul. Quand le statut devient `completed`, la collecte télécharge les réponses, les valide et écrit `predictions.json`, `manifest.csv` et les labels YOLO.

In [ ]:
state_path = config.output_root / 'state.json'
if state_path.exists() and json.loads(state_path.read_text()).get('batch_id'):
    state = refresh_status(config)
else:
    state = None
    display(Markdown('Aucun batch soumis pour ce run.'))

In [ ]:
if state and state['batch_status'] == 'completed':
    summary = collect_batch(config)
    display(summary)
elif state:
    display(Markdown(f"Batch encore **{state['batch_status']}** : reviens exécuter ces cellules plus tard."))

## 5. Inspection visuelle sobre

La boîte est fine et aucun numéro ne masque le ballon. Les images sont relues directement depuis la vidéo : aucune galerie JPEG supplémentaire n'est stockée.

In [ ]:
import cv2
import matplotlib.pyplot as plt

predictions_path = config.output_root / 'predictions.json'
if predictions_path.exists():
    predictions = json.loads(predictions_path.read_text())['predictions']
    shown = predictions[:12]
    capture = cv2.VideoCapture(str(config.video_path))
    fig, axes = plt.subplots(3, 4, figsize=(16, 9))
    for axis, prediction in zip(axes.flat, shown):
        capture.set(cv2.CAP_PROP_POS_FRAMES, prediction['source_frame'])
        ok, frame = capture.read()
        if not ok:
            axis.set_visible(False)
            continue
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axis.imshow(image)
        bbox = prediction['bbox_current_pixels']
        if bbox:
            x1, y1, x2, y2 = bbox
            axis.add_patch(plt.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor='#00ff66', linewidth=1.2,
            ))
        state_label = 'ballon' if bbox else 'négatif'
        axis.set_title(
            f"frame {prediction['source_frame']} — {state_label} — "
            f"conf {prediction['confidence']:.2f}", fontsize=9
        )
        axis.axis('off')
    capture.release()
    for axis in axes.flat[len(shown):]:
        axis.set_visible(False)
    fig.tight_layout()
    plt.show()
else:
    display(Markdown('Aucun résultat collecté pour le moment.'))

## 6. Nettoyer les fichiers distants

À faire seulement après avoir vérifié que `predictions.json` existe et s'ouvre correctement. Les résultats locaux et la vidéo ne sont pas supprimés.

In [ ]:
DELETE_REMOTE_FILES = False

if DELETE_REMOTE_FILES:
    deleted = cleanup_remote_files(config)
    display(Markdown(f"{deleted:,} fichiers API supprimés."))
else:
    display(Markdown('Nettoyage distant désactivé.'))

## 7. Préparer les six nouvelles vidéos — uniquement hors ligne

Cette section est indépendante du batch ci-dessus : elle ne charge pas la clé et n’appelle ni Luna ni un autre modèle. On extrait les images candidates avant de les annoter.

**Pas retenu : une image sur quatre**, sans plafond de 100 images. Résolution d’origine, JPEG qualité 95. Attention : à 30 FPS cela donne 7,5 images/seconde, contre 15 à 60 FPS ; ce n’est pas une cadence temporelle uniforme.

Chaque image garde sa vidéo, son auteur, son URL, son numéro de frame et son timestamp de présentation. Les identifiants incluent la vidéo : les frames de deux clips ne peuvent pas se remplacer.

Les colonnes match_id, scene_id et split sont encore non renseignées. Une vidéo n’est pas automatiquement un match indépendant : il faudra regrouper les extraits et ralentis d’un même match avant de créer les splits. Aucune annotation vide n’est produite : **non annoté ne signifie pas absence de ballon**.

La cellule peut être relancée avec les mêmes paramètres sans recopier les images. Pour changer le pas, utiliser un nouveau dossier de sortie ; aucun travail existant n’est écrasé.

In [ ]:
import csv
import json
from pathlib import Path

from IPython.display import Markdown, display

from pitchlens.data.extract_video_frames import extract_video_frames

NEW_PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
NEW_CONFIG = json.loads((NEW_PROJECT_ROOT / "configs/new_video_frames.json").read_text())
FRAME_STRIDE = NEW_CONFIG["frame_stride"]
JPEG_QUALITY = NEW_CONFIG["jpeg_quality"]
NEW_OUTPUT_ROOT = NEW_PROJECT_ROOT / NEW_CONFIG["output_root"]
new_report = extract_video_frames(
    project_root=NEW_PROJECT_ROOT,
    download_manifest=NEW_PROJECT_ROOT / NEW_CONFIG["download_manifest"],
    output_root=NEW_OUTPUT_ROOT,
    frame_stride=FRAME_STRIDE,
    jpeg_quality=JPEG_QUALITY,
)
with (NEW_OUTPUT_ROOT / "manifest.csv").open() as stream:
    new_frames = list(csv.DictReader(stream))
lines = ["| Vidéo | Auteur | FPS | Images retenues |", "|---|---|---:|---:|"]
for video in new_report["videos"]:
    lines.append(f"| {video['video_id']} | {video['uploader']} | {video['fps']:.2f} | {video['images']} |")
display(Markdown("\n".join(lines)))
display(Markdown(
    f"**Total : {new_report['images']} images non annotées** — "
    f"{new_report['image_bytes']/1024**2:.1f} Mio — aucun appel API.\n\n"
    f"Sortie : {NEW_OUTPUT_ROOT}"
))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as DisplayImage
from PIL import Image as PILImage

fig, axes = plt.subplots(2, 3, figsize=(9, 11))
for ax, video in zip(axes.flat, new_report["videos"]):
    candidates = [row for row in new_frames if row["video_id"] == video["video_id"]]
    row = candidates[len(candidates)//2]
    with PILImage.open(NEW_PROJECT_ROOT / row["image_path"]) as image:
        image.thumbnail((300, 540))
        pixels = np.asarray(image).copy()
    ax.imshow(pixels)
    ax.set_title(f"{video['uploader']}\n{video['video_id']}\nFrame {row['source_frame']} · {float(row['timestamp_seconds']):.2f} s", fontsize=8)
    ax.axis("off")
fig.suptitle("Une image par nouvelle vidéo — aucune boîte proposée")
fig.tight_layout()
fig.subplots_adjust(hspace=.3)
preview_path = NEW_OUTPUT_ROOT / "preview.png"
fig.savefig(preview_path, dpi=110)
plt.close(fig)
display(DisplayImage(filename=str(preview_path)))

Les 782 images sont envoyées à GPT-6 Luna en raisonnement **max**, avec trois images originales (avant/actuelle/après), JPEG 95 et un plafond de **32 768 tokens par réponse, réflexion comprise**. Ce plafond n'est pas une consommation imposée. Configuration : `configs/luna_new_videos_20260928.json`. Six lots séparés empêchent les voisins temporels de franchir une limite de vidéo. Le processus tourne hors notebook, récupère les réponses et ne relance pas automatiquement les échecs payants. Les propositions restent à revoir : rien n'est ajouté à luna_v001 ni utilisé pour entraîner.

In [ ]:
# Lecture locale seulement : aucun appel API ni lancement payant.
import json
from pathlib import Path
from IPython.display import Markdown, display
annotation_root = Path('data/interim/gpt6_luna_new_videos_20260928')
lines = ['| Vidéo | Images envoyées | Statut | Réponses |', '|---|---:|---|---|']
for plan_path in sorted(annotation_root.glob('*/plan.json')):
    plan = json.loads(plan_path.read_text())
    state_path = plan_path.parent / 'state.json'
    state = json.loads(state_path.read_text()) if state_path.exists() else {}
    uploaded = len(state.get('uploaded_images', {}))
    required = plan['unique_uploaded_frame_count']
    status = state.get('batch_status') or 'envoi en cours / en attente'
    counts = state.get('request_counts', {})
    lines.append(f'| {plan_path.parent.name} | {uploaded}/{required} | {status} | {counts} |')
display(Markdown('\n'.join(lines)))
summary_path = annotation_root / 'summary.json'
if summary_path.exists():
    display(json.loads(summary_path.read_text()))
log_path = annotation_root / 'run.log'
if log_path.exists():
    print('\n'.join(log_path.read_text().splitlines()[-12:]))

## 8. Pilote de précision : zoom seul ou zoom avec repères
20 images positives réparties entre les six vidéos, 40 réponses GPT-6 Luna max. Les propositions initiales servent uniquement à choisir une zone large ; on ne les présente pas comme une vérité. Le zoom conserve une marge, est agrandi et reçoit une bordure blanche identique dans les deux variantes. Les repères sont fins, les chiffres restent dans cette bordure, et la reconversion des coordonnées est faite par le code.
Budget autorisé : 0,50 USD hors taxes. Avant soumission, l’API compte les entrées ; le calcul suppose que chaque réponse pourrait utiliser les 32 768 tokens autorisés et que toutes les entrées pourraient être facturées en écriture de cache. Le batch est refusé si cette estimation maximale dépasse le budget. Aucun retry payant automatique.
Il n’y avait pas de corrections humaines soumises lors du lancement : c’est une comparaison visuelle, **pas une mesure de précision**. Le projet Label Studio et ses annotations ne sont jamais remplacés.

In [ ]:
# Lecture locale uniquement. PAGE peut valoir 1 à 5.
import json
from pathlib import Path
from IPython.display import Image, display
PAGE = 1
pilot_root = Path('data/interim/gpt6_luna_crop_pilot20_20260929')
for name in ['state.json', 'summary.json']:
    path = pilot_root / name
    if path.exists():
        obj = json.loads(path.read_text())
        if name == 'state.json':
            display({key: obj.get(key) for key in ['batch_status', 'request_counts', 'budget_check']})
        else:
            display(obj)
comparison = pilot_root / f'comparison_page_{PAGE:02d}.png'
if comparison.exists():
    display(Image(filename=str(comparison)))
else:
    print('Comparaisons pas encore disponibles. Ne pas relancer les cellules API du notebook.')